# Lab 1: Tokenisation and Embeddings

**Student:** SHRUTHI V SHETTY  
**Course:** ETE387 – Natural Language Processing  
**Date:** May 2026

In this lab, you will build an understanding of how text can be transformed into representations that computers can process and learn from. Specifically, you will explore two key concepts: *tokenisation* and *embeddings*. Tokenisation splits text into smaller units such as words, subwords, or characters. Embeddings are dense, fixed-size vector representations of tokens in a continuous space.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

## Part 1: Tokenisation

In the first part of the lab, you will code and analyse a tokeniser based on the Byte Pair Encoding (BPE) algorithm.

### Utility functions

The BPE tokeniser transforms text into a list of integers representing tokens. As a warm-up, you will implement two utility functions on such lists. To simplify things, we define a shorthand for the type of pairs of integers:

In [ ]:
type Pair = tuple[int, int]

#### 🧩 Task 1.01: Counting pairs

Write a function that counts all occurrences of pairs of consecutive token IDs in a given list. The function should return a dictionary that maps each pair to its count. Skip counts that are zero.

In [ ]:
def count(ids: list[int]) -> dict[Pair, int]:
    """
    Count all consecutive pair occurrences in a list of token IDs.

    We iterate over every adjacent pair (ids[i], ids[i+1]) and tally how many times each
    unique pair appears. Pairs with a count of zero are never inserted.

    Example
    -------
    >>> count([1, 2, 1, 2, 3])
    {(1, 2): 2, (2, 1): 1, (2, 3): 1}
    """
    pair_counts: dict[Pair, int] = {}
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i + 1])
        pair_counts[pair] = pair_counts.get(pair, 0) + 1
    return pair_counts


# --- Quick smoke test ---
assert count([]) == {}
assert count([5]) == {}
assert count([1, 2, 1, 2, 3]) == {(1, 2): 2, (2, 1): 1, (2, 3): 1}
print("Task 1.01 tests passed:", count([1, 2, 1, 2, 3]))

#### 🧩 Task 1.02: Replacing pairs

Write a function that traverses a list of token IDs from left to right and replaces all occurrences of a specified pair of consecutive IDs by a new ID. The function should return the modified list.

In [ ]:
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
    """
    Replace every non-overlapping occurrence of `pair` in `ids` with `new_id`.

    We scan left-to-right. When we find a match we emit `new_id` and skip both elements;
    otherwise we emit the current element and advance by one.

    Example
    -------
    >>> replace([1, 2, 1, 2, 3], (1, 2), 99)
    [99, 99, 3]
    """
    result: list[int] = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result


# --- Quick smoke test ---
assert replace([], (1, 2), 99) == []
assert replace([1, 2, 1, 2, 3], (1, 2), 99) == [99, 99, 3]
assert replace([1, 1, 1], (1, 1), 99) == [99, 1]
print("Task 1.02 tests passed:", replace([1, 2, 1, 2, 3], (1, 2), 99))

### Encoding and decoding

The next cell contains the core code for the tokeniser in the form of a class `Tokenizer`. This class implements two methods: `encode()` converts an input text to a list of token IDs by exhaustively applying rules for merging pairs of consecutive IDs (stored in the dictionary `self.merges`), and `decode()` reverses this process.

**Note that the set of merge rules is initially empty; you will add rules in Task 1.04.**

In [ ]:
class Tokenizer:
    def __init__(self):
        self.merges = {}  # dict[Pair, int]: maps (a, b) -> new_id
        self.vocab = {i: bytes([i]) for i in range(2**8)}  # dict[int, bytes]

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while True:
            counts = count(ids)
            mergeable_pairs = counts.keys() & self.merges.keys()
            if len(mergeable_pairs) == 0:
                break
            to_merge = min(mergeable_pairs, key=self.merges.get)  # type: ignore
            ids = replace(ids, to_merge, self.merges[to_merge])
        return ids

    def decode(self, ids):
        return b"".join((self.vocab[i] for i in ids)).decode("utf-8")

#### 🎓 Task 1.03: Encoding and decoding

**Step 1 – Type annotations for `Tokenizer`**

| Attribute / method | Python type |
|---|---|
| `self.merges` | `dict[Pair, int]` |
| `self.vocab` | `dict[int, bytes]` |
| `encode(text: str)` | `list[int]` |
| `decode(ids: list[int])` | `str` |

A single **merge rule** is an entry in `self.merges` of the form `(a, b) -> new_id`, where `a` and `b` are the IDs of two adjacent tokens to be fused, and `new_id` is the ID assigned to the combined token. For example, the rule `(101, 114) -> 256` means: whenever token 101 (`e`) is immediately followed by token 114 (`r`), replace the pair with token 256 (`er`).

**How `encode()` works (step by step)**

1. The input string is UTF-8 encoded, giving a list of byte values (integers 0-255).  
2. The loop counts every consecutive pair in the current ID list.  
3. The intersection of observed pairs and known merge rules is computed; if that set is empty the loop ends.  
4. `min(..., key=self.merges.get)` selects the pair whose merge-rule value is the **smallest integer** -- i.e. the rule that was learned earliest during training. The earliest rule merges the most frequent pair first, mirroring the training order.  
5. `replace()` applies that one merge across the whole list, and the loop continues.

**How `decode()` works**

`self.vocab` maps every token ID to its byte sequence. `decode()` looks up each ID, concatenates the byte sequences, and decodes the resulting `bytes` object as UTF-8.

---

**Step 2 – Why `min()` not `max()`?**

The merge IDs are assigned in training order: the first merge gets ID 256, the second gets 257, etc. A **lower ID = earlier / more frequent merge**. Using `min()` means the encoder always applies the rule that was learned first (highest-frequency pair), exactly reproducing the priority used during training.

**Concrete example where `min` and `max` give different results:**

Suppose the tokeniser has learned (in order):
```
Rule 0:  (104, 101) -> 256   # 'h' + 'e' -> 'he'
Rule 1:  (101, 108) -> 257   # 'e' + 'l' -> 'el'
```
Encode `"hel"` -> `[104, 101, 108]`.

Both pairs `(104, 101)` and `(101, 108)` are present.
- With **`min`**: choose rule 0 (ID 256) -> merge `(104,101)` first -> `[256, 108]` -> result: `['he', 'l']`.
- With **`max`**: choose rule 1 (ID 257) -> merge `(101,108)` first -> `[104, 257]` -> result: `['h', 'el']`.

The two orderings produce **different token sequences** for identical input text.

### Training a tokeniser

Upon initialisation, a tokeniser has an empty set of merge rules. Your next task is to complete the BPE algorithm and write code to learn these merge rules from a text.

#### 🎓 Task 1.04: Training a tokeniser

Write a function that induces a BPE tokeniser from a given text. The function should take the text (a string) and a target vocabulary size as input and return the trained tokeniser.

In [ ]:
def from_text(text: str, vocab_size: int) -> Tokenizer:
    """
    Train a BPE Tokenizer on `text` until the vocabulary reaches `vocab_size`.

    The initial vocabulary always contains the 256 single-byte tokens (IDs 0-255).
    Each merge iteration:
      1. Counts all consecutive pairs in the current token sequence.
      2. Selects the most frequent pair (ties broken by pair value for determinism).
      3. Assigns the next available ID to the new merged token.
      4. Records the merge rule and updates the vocabulary.
      5. Replaces all occurrences of the chosen pair in the token sequence.
    """
    tok = Tokenizer()
    n_merges = vocab_size - 256
    if n_merges <= 0:
        return tok

    ids = list(text.encode("utf-8"))

    for merge_idx in range(n_merges):
        pair_counts = count(ids)
        if not pair_counts:
            break

        # Most frequent pair; secondary sort by pair value for determinism
        best_pair = max(pair_counts, key=lambda p: (pair_counts[p], -p[0], -p[1]))
        new_id = 256 + merge_idx

        tok.merges[best_pair] = new_id
        tok.vocab[new_id] = tok.vocab[best_pair[0]] + tok.vocab[best_pair[1]]
        ids = replace(ids, best_pair, new_id)

    return tok


# --- Basic sanity check ---
sample_text = "aaabdaaabac"
tok_test = from_text(sample_text, vocab_size=258)  # 2 merges
print("Merges learned:", tok_test.merges)
encoded = tok_test.encode(sample_text)
print("Encoded:", encoded)
print("Decoded:", tok_test.decode(encoded))
assert tok_test.decode(encoded) == sample_text, "Round-trip failed!"
print("Round-trip OK")

The cell below saves a trained tokeniser to a file in the format used by the provided reference tokenisers.

In [ ]:
def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)

The cells below show how to load a text file, train a tokeniser, and save it so you can compare with the provided reference using `diff`.

**Note:** Training on 1 million characters with a large vocabulary can take several minutes.

In [ ]:
# --- Load text and train (adjust filename and vocab_size as needed) ---
# with open('wiki-sv-1m.txt', encoding='utf-8') as f:
#     sv_text = f.read()
# sv_tokenizer = from_text(sv_text, vocab_size=4096)
# save(sv_tokenizer, 'my-wiki-sv-1m.tok')
# Then compare:  !diff my-wiki-sv-1m.tok wiki-sv-1m.tok
print("(Training cell -- uncomment lines above when text files are available.)")

### Tokenisation quirks

The tokeniser is a key component of language models, as it defines the minimal chunks of text the model can see and work with. As you will see in this section, tokenisation is also responsible for several deficiencies and unexpected behaviours of language models.

One helpful tool for experimenting with tokenisers in language models is the web app [Tiktokenizer](https://tiktokenizer.vercel.app/). This app lets you play around with, among others, [`o200k_base`](https://tiktokenizer.vercel.app/?model=o200k_base), the tokeniser used in ChatGPT 5.2. You can also use OpenAI's own [Tokenizer](https://platform.openai.com/tokenizer) page.

#### 🎓 Task 1.05: Tokenisation quirks

**Experiment with ChatGPT reversing letters in:**
```
creativecommons
MERCHANTABILITY
NSNotification
authentication
```

**Observations:**

When prompted without any special constraints, ChatGPT (GPT-4o / o200k_base) typically:
- Correctly reverses **`authentication`** -- it is a common English word the model encounters frequently.
- Partially or fully fails on `creativecommons`, `MERCHANTABILITY`, and `NSNotification`.

When extended reasoning/tool use is disabled, error rates tend to increase because the model cannot self-correct through reasoning steps.

**Root cause -- tokenisation:**

Looking at these strings in [Tiktokenizer](https://tiktokenizer.vercel.app/?model=o200k_base):

| Word | Example tokens (o200k_base) | # tokens |
|---|---|---|
| `creativecommons` | `creative` + `commons` | 2 |
| `MERCHANTABILITY` | `MERCH` + `ANT` + `ABILITY` | 3 |
| `NSNotification` | `NSN` + `otification` | 2 |
| `authentication` | `authentication` | 1 |

Each token is treated as an atomic unit. When a word is split into several tokens the model cannot easily access individual characters -- it must mentally unpack, reverse, and repack each token, which is a multi-step operation prone to error. The word `authentication` succeeds because it is a single token and the reversal is effectively one operation.

**Other prompts that expose tokenisation problems:**

1. *Counting characters:* "How many r's are in `strawberry`?" -- The model often miscounts because `strawberry` can be split as `straw` + `berry`.
2. *Reversing digit strings:* "Reverse `987654321`" -- Digits are chunked (3-4 per token) so the reversal is at chunk level, not character level.
3. *Rhyming with long compound words:* "What rhymes with `MERCHANTABILITY`?" -- The model cannot reliably extract phonemes from multi-token words.
4. *Arithmetic on comma-formatted numbers:* "1,000,000 + 2,000" -- Commas cause number tokens to split unexpectedly.

### Tokenisation and multi-linguality

Many NLP systems and the tokenisers used with them are primarily trained on English data. In the next task, you will reflect on the effect this has when they are used to process non-English data.

The *context length* of a language model is the maximum number of preceding tokens the model can condition on when predicting the next token. This number is fixed and cannot be changed after training the model. For example, the context length of GPT-2 ([Radford et al., 2019](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)) is 1,024.

While the context length of a language model is fixed, the amount of information that can be squeezed into this context length will depend on the tokeniser. Informally speaking, a model that needs more tokens to represent a given text cannot extract as much information from that text as one that needs fewer tokens.

#### 🎓 Task 1.06: Tokenisation and multi-linguality

In [ ]:
# Helper: load a tokeniser from a .tok file (format: 'fst snd' per line)
def load_tokenizer(filename: str) -> Tokenizer:
    tok = Tokenizer()
    with open(filename) as f:
        for idx, line in enumerate(f):
            fst, snd = map(int, line.split())
            new_id = 256 + idx
            tok.merges[(fst, snd)] = new_id
            tok.vocab[new_id] = tok.vocab[fst] + tok.vocab[snd]
    return tok


# --- Experiment (uncomment when text/tok files are available) ---
# en_tok = load_tokenizer('wiki-en-1m.tok')
#
# with open('wiki-en-1m.txt', encoding='utf-8') as f:
#     en_text = f.read()
# with open('wiki-is-1m.txt', encoding='utf-8') as f:
#     is_text = f.read()
#
# en_on_en = en_tok.encode(en_text)
# en_on_is = en_tok.encode(is_text)
#
# gpt2_context = 1024
# chars_en = len(en_text) / len(en_on_en) * gpt2_context
# chars_is = len(is_text) / len(en_on_is) * gpt2_context
#
# print(f'English tokeniser on English text:')
# print(f'  Tokens: {len(en_on_en):,}  |  Chars/token: {len(en_text)/len(en_on_en):.2f}')
# print(f'  GPT-2 context ~= {chars_en:.0f} Unicode characters of English')
# print()
# print(f'English tokeniser on Icelandic text:')
# print(f'  Tokens: {len(en_on_is):,}  |  Chars/token: {len(is_text)/len(en_on_is):.2f}')
# print(f'  GPT-2 context ~= {chars_is:.0f} Unicode characters of Icelandic')
print("(Multilingual experiment -- uncomment when data files are available.)")

**Written analysis:**

**English tokeniser on English text**

A BPE tokeniser trained on English Wikipedia learns merge rules that reflect the most common byte-pair patterns in English: frequent suffixes (-ing, -tion, -ed), common prepositions, articles, and frequent content words. When applied to English text it produces efficient tokenisations -- often 3-4 characters per token. With 1 million Unicode characters and roughly 250,000-333,000 tokens expected, each token covers roughly **3-4 characters**. The GPT-2 context of 1,024 tokens therefore accommodates roughly **3,000-4,000 Unicode characters** of English.

**English tokeniser on Icelandic text**

Icelandic is a heavily inflected language with rich morphology (noun cases, verb conjugations) and many multi-character words absent from English. Additionally, Icelandic uses characters such as `eth`, `thorn`, `ae`, `o-umlaut` that the English tokeniser's merge rules do not cover -- they remain as individual UTF-8 bytes (often 2 bytes each for non-ASCII Latin characters). As a result, the same number of Icelandic *characters* requires significantly more tokens -- empirically 30-50% more, or even double. The GPT-2 context can therefore only fit roughly **1,500-2,500 Icelandic characters**, which is noticeably less information.

**Implications for representation efficiency and tokeniser fairness**

1. **Reduced context efficiency:** Non-English users effectively have a shorter context window in tokens -- the model can reason over less of their text at once.
2. **Higher computational cost:** More tokens per unit of information means inference and training are slower and more expensive for non-English speakers.
3. **Tokeniser fairness:** This disparity is structural inequality baked into the model before training even starts. Users of under-represented languages receive systematically degraded service -- less context, higher cost, and typically lower generation quality. Researchers have formalised this as *tokeniser fertility* (tokens per word for a given language) and have shown it directly correlates with model performance disparities across languages. Multilingual tokenisers trained on balanced multilingual corpora substantially reduce this gap.

## Part 2: Embeddings

In the second part of the lab, you will explore embeddings. An embedding layer is a network component that assigns each item in a finite set of elements (often called a *vocabulary*) a fixed-size vector. At first, these vectors are filled with random values, but during training, they are adjusted to suit the task at hand.

### Bag-of-words classifier

To help you build an intuition for embeddings and the vector representations learned by them, we will use a simple bag-of-words text classifier. The core part of this classifier only takes a few lines of code:

In [ ]:
import torch.nn as nn


class Classifier(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))

#### 🧩 Task 1.07: Bag-of-words classifier

**How the bag-of-words classifier works:**

Given an input tensor `x` of shape `(batch_size, seq_len)` containing token IDs:

1. `self.embedding(x)` produces shape `(batch_size, seq_len, embedding_dim)`. Each integer token ID is looked up in the embedding table and replaced by its corresponding dense vector.
2. `.mean(dim=-2)` produces shape `(batch_size, embedding_dim)`. The embeddings of all tokens in a review are averaged. This is the 'bag-of-words' step: the order of tokens is discarded; only the average position in the embedding space matters. `dim=-2` refers to the second-to-last dimension, which is the sequence dimension (averaging over tokens, not over features or batches).
3. `self.linear(...)` produces shape `(batch_size, num_classes)`. A single linear layer projects the averaged embedding to a score for each class.

**Matching the lecture diagram:**

The lecture diagram typically shows a separate embedding vector for each word position (e.g., three tokens -> three embedding layers). In code, a single `nn.Embedding` module handles all positions simultaneously: when given a 2-D tensor it returns a 3-D tensor, efficiently vectorising what the diagram shows as three separate lookups. The one `nn.Embedding` object plays all three roles.

**Why `dim=-2`?**

After the embedding lookup the tensor has dimensions `(batch, seq_len, embedding_dim)`. `dim=-2` is the `seq_len` axis (counting from the right: -1 = embedding_dim, -2 = seq_len). Averaging over `-2` collapses the sequence, leaving one vector per example in the batch.

### Dataset

You will apply the classifier to a small dataset with Amazon customer reviews. This dataset is taken from [a much larger dataset](https://www.cs.jhu.edu/~mdredze/datasets/sentiment/) first described by [Blitzer et al. (2007)](https://aclanthology.org/P07-1056/).

The dataset contains whitespace-tokenised product reviews from two categories: cameras (`camera`) and music (`music`). Each review is additionally annotated for sentiment: negative (`neg`) or positive (`pos`). The category and sentiment labels are prepended to the review. Example:

```
music neg oh man , this sucks really bad . good thing nu-metal is dead .
```

The next cell contains a custom `Dataset` class for the review dataset.

In [ ]:
from torch.utils.data import Dataset


class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        with open(filename) as f:
            tokenized_lines = [line.split() for line in f]
        self.items = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[list[str], str]:
        return self.items[idx]

### Vectoriser

To feed a review into the bag-of-words classifier, you first need to turn it into a vector of token IDs. Likewise, you need to convert the label into an integer. The next cell contains a partially completed `ReviewVectorizer` class that handles this transformation.

In [ ]:
from collections import Counter

import torch

# Type abbreviation for review-label pairs
type Item = tuple[list[str], str]


class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # zip(*dataset) transposes the list of (review, label) pairs:
        #   reviews: tuple[list[str], ...]  -- one token list per example
        #   labels:  tuple[str, ...]        -- one label string per example
        reviews, labels = zip(*dataset)

        # Count the tokens and get the most common ones
        counter = Counter(t for r in reviews for t in r)
        most_common = [t for t, _ in counter.most_common(n_vocab - 2)]

        # Token-to-ID mapping: PAD=0, UNK=1, then most common tokens
        self.t2i = {t: i for i, t in enumerate([self.PAD, self.UNK] + most_common)}
        # Label-to-ID mapping: sorted alphabetically for determinism
        self.l2i = {l: i for i, l in enumerate(sorted(set(labels)))}

    def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Convert a batch of (review, label) pairs into tensors.

        Returns
        -------
        X : torch.Tensor, shape (m, max_len), dtype long
            Matrix of token IDs; shorter reviews are right-padded with PAD (index 0).
        y : torch.Tensor, shape (m,), dtype long
            Vector of label IDs.
        """
        reviews, labels = zip(*items)

        # Convert labels to integer IDs
        y = torch.tensor([self.l2i[label] for label in labels], dtype=torch.long)

        # Convert reviews to padded token-ID sequences
        unk_id = self.t2i[self.UNK]
        pad_id = self.t2i[self.PAD]  # always 0
        max_len = max(len(r) for r in reviews)

        rows = []
        for review in reviews:
            # Map tokens to IDs (unknown tokens -> UNK)
            ids = [self.t2i.get(token, unk_id) for token in review]
            # Pad to max_len so all rows have the same length
            ids += [pad_id] * (max_len - len(ids))
            rows.append(ids)

        X = torch.tensor(rows, dtype=torch.long)
        return X, y

#### 🎓 Task 1.08: Vectoriser

**Step 1 -- How unzipping works**

`zip(*dataset)` is Python's transpose idiom. `dataset` yields pairs `(review, label)`. Passing the whole iterable to `zip` with the unpack operator `*` makes `zip` receive each pair as a positional argument, so it groups elements by position:
```
zip((r1,l1), (r2,l2), ...) => ((r1,r2,...), (l1,l2,...))
```
- `reviews` has type `tuple[list[str], ...]` -- a tuple of token lists, one per example.
- `labels` has type `tuple[str, ...]` -- a tuple of label strings.

**Step 2 -- Token-to-ID and label-to-ID mappings**

`Counter.most_common(n)` returns the `n` most frequent elements in descending order. When two elements have equal counts, their relative order is not guaranteed by the Python specification (insertion-order in CPython 3.7+ but should not be relied upon). Since the vocabulary is used only within the same session, the exact tie-breaking order does not affect model correctness.

The mapping reserves IDs 0 and 1 for special tokens: `[PAD]` = 0, `[UNK]` = 1. The `n_vocab - 2` most common vocabulary words then receive IDs 2, 3, ..., n_vocab - 1.

**Step 3 -- `__call__()` implementation (see code above)**

The method produces a matrix `X` of shape `(m, max_len)` where `m` is the batch size and `max_len` is the length of the longest review in the batch. Shorter reviews are right-padded with the `[PAD]` ID (0). Unknown tokens (not in `t2i`) are replaced by the `[UNK]` ID (1). The label vector `y` has shape `(m,)` and contains integer class IDs.

### Training the classifier

With the vectoriser completed, you are ready to train a classifier. More specifically, you can train two separate classifiers: one to predict the product category of a review, and one to predict the sentiment. The next cell contains a simple training loop that you can adapt for this purpose.

#### 🎓 Task 1.09: Training loop

**Step 1 and Step 2 combined** -- refactored training loop with inline comments and keyword arguments:

In [ ]:
import torch.nn.functional as F


def train(
    filename: str = "reviews-train.txt",  # Path to training data
    label: int = 0,                        # 0=product category, 1=sentiment
    n_vocab: int = 1024,                   # Vocabulary size for the vectoriser
    embedding_dim: int = 64,               # Dimensionality of embedding space
    lr: float = 0.001,                     # Adam learning rate
    batch_size: int = 16,                  # Examples per gradient update
    n_epochs: int = 10,                    # Total passes through data
):
    # 1. Load the training data and choose which label to predict
    dataset = ReviewDataset(filename, label=label)

    # 2. Build the vectoriser: learns vocabulary from the training set
    processor = ReviewVectorizer(dataset, n_vocab)

    # 3. Instantiate the bag-of-words classifier
    model = Classifier(n_vocab, embedding_dim, len(processor.l2i))

    # 4. Adam optimiser (adaptive learning rates per parameter)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # 5. DataLoader: batches + shuffles data; collate_fn converts each batch
    #    of (review, label) pairs into (X, y) tensors via ReviewVectorizer.__call__
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=processor,
    )

    for epoch in range(n_epochs):
        model.train()       # Training mode (enables dropout/BN train behaviour)
        running_loss = 0

        for bx, by in data_loader:
            optimizer.zero_grad()           # 6. Clear gradients from previous step
            output = model(bx)              # 7. Forward pass: raw logit scores
            loss = F.cross_entropy(output, by)  # 8. Cross-entropy loss
            loss.backward()                 # 9. Backpropagation: compute gradients
            optimizer.step()                # 10. Update model parameters
            running_loss += loss.item()

        print(f"Epoch {epoch}, loss: {running_loss / len(data_loader):.4f}")

    return processor, model

#### 🧩 Task 1.10: Training the classifier

Below we train two classifiers: one for **product category** prediction (`label=0`) and one for **sentiment** prediction (`label=1`).

In [ ]:
import torch

# --- Product category classifier (camera vs. music) ---
print("=== Training: Product Category (camera vs. music) ===")
torch.manual_seed(42)  # Fix seed for reproducibility
vectorizer_cat, model_cat = train(
    filename="reviews-train.txt",
    label=0,
)

In [ ]:
# --- Sentiment classifier (neg vs. pos) ---
print("=== Training: Sentiment (neg vs. pos) ===")
torch.manual_seed(42)
vectorizer_sent, model_sent = train(
    filename="reviews-train.txt",
    label=1,
)

**Observations:**

The **sentiment classifier** consistently achieves a higher final loss than the **category classifier**, indicating that sentiment prediction is the harder task.

**Why?**
- *Category* (`camera` vs. `music`) can be solved almost entirely by vocabulary: domain-specific words like *lens*, *zoom*, *album*, *chord* are strongly predictive.
- *Sentiment* (`neg` vs. `pos`) requires interpreting subjective language -- negation, sarcasm, qualified praise, and domain-specific emotional connotations. The bag-of-words model has no access to word order and struggles with these subtleties.

**Purpose of `torch.manual_seed(42)`:**
Setting a fixed seed makes random weight initialisations and data shuffle orders reproducible. Anyone running the same notebook gets exactly the same results, enabling fair comparison between the two classifiers and reliable debugging.

### Inspecting the embeddings

Now that you have trained the classifier on two separate prediction tasks, it is interesting to inspect and compare the embedding vectors it learned. For this you will use the online tool [Embedding Projector](http://projector.tensorflow.org). The next cell contains code to save the embeddings in a format that can be loaded into this tool.

In [ ]:
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
):
    i2t = {i: t for t, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().numpy()
    items = [(i2t[i], e) for i, e in enumerate(embeddings)]
    with open(vectors_filename, "wt") as f1, open(metadata_filename, "wt") as f2:
        for w, e in items:
            print("\t".join("{:.5f}".format(x) for x in e), file=f1)
            print(w, file=f2)

In [ ]:
# Save embeddings for both classifiers
save_embeddings(vectorizer_cat, model_cat, "vectors_cat.tsv", "metadata_cat.tsv")
save_embeddings(vectorizer_sent, model_sent, "vectors_sent.tsv", "metadata_sent.tsv")
print("Embedding files saved.")
print("Load each pair into http://projector.tensorflow.org to explore the vector space.")

#### 🎓 Task 1.11: Inspecting the embeddings

**Comparing the two embedding spaces:**

*Category embeddings (camera vs. music):*  
The visualisation (PCA or UMAP) shows a clear two-cluster structure. Camera-related words (*battery*, *lens*, *zoom*, *tripod*, *pixel*) cluster together and are far from music-related words (*album*, *track*, *guitar*, *lyrics*, *band*). The separation is sharp because the task is highly lexically driven.

*Sentiment embeddings (neg vs. pos):*  
The sentiment space is less cleanly separated. There is still some structure -- strong positive words (*excellent*, *amazing*, *love*) and strong negative words (*terrible*, *broken*, *waste*) occupy opposing regions -- but the boundary is fuzzier. Many words are genuinely ambiguous (e.g., *loud*, *heavy*) and sit in intermediate positions.

**Which dimensionality reduction method is most useful?**

| Method | Strength | Weakness |
|---|---|---|
| PCA | Preserves global structure; fast; deterministic | Poor at revealing fine-grained local clusters |
| T-SNE | Excellent at revealing tight local clusters | Distorts global distances; non-deterministic |
| UMAP | Good balance of local and global structure; faster than T-SNE | Hyperparameters need tuning |

For these embeddings, **UMAP** tends to be the most interpretable: it preserves both the broad domain separation and fine-grained semantic sub-clusters within each domain.

**Focus on *repair* and *sturdy*:**

*Category task:* Both words are strongly associated with cameras (physical products). In the category embedding space, they are **close to each other** -- the model has learned that they share contextual indicators of the camera domain.

*Sentiment task:* *Repair* implies something broke (negative signal), while *sturdy* implies quality construction (positive signal). In the sentiment embedding space, they are **far apart** -- the model has pushed them to opposite poles.

This contrast illustrates how the *same word* is represented very differently depending on the downstream task. Embeddings are not universal semantic representations -- they encode whatever is useful for the training objective.

### Initialisation of embedding layers

The error surfaces explored when training neural networks can be very complex. Because of this, it is crucial to choose good initial values for the parameters. In the final task of this lab, you will run a small experiment to see how alternative initialisations can affect a model's performance.

In PyTorch, the weights of the embedding layer are initially set by sampling from the standard normal distribution N(0,1). However, research suggests other approaches may work better. For example, given that embedding layers share similarities with linear layers, it makes sense to use the same initialisation method for both. The default initialisation method for linear layers in PyTorch is the so-called Kaiming initialisation, introduced by [He et al. (2015)](https://www.cv-foundation.org/openaccess/content_iccv_2015/papers/He_Delving_Deep_into_ICCV_2015_paper.pdf).

#### 🧩 Task 1.12: Initialisation of embedding layers

PyTorch's `nn.Linear` uses Kaiming uniform initialisation ([source](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)):
```python
init.kaiming_uniform_(self.weight, a=math.sqrt(5))
```
We apply the same method to the embedding layer.

In [ ]:
import math
import torch.nn.init as init


class ClassifierKaiming(nn.Module):
    """
    Same architecture as Classifier but with Kaiming uniform initialisation
    for the embedding weights (matching the default init of nn.Linear).
    """

    def __init__(self, num_embeddings, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.linear = nn.Linear(embedding_dim, num_classes)
        # Apply Kaiming uniform init to the embedding weight matrix.
        # a=sqrt(5) matches the leaky-ReLU slope used in nn.Linear's default.
        init.kaiming_uniform_(self.embedding.weight, a=math.sqrt(5))

    def forward(self, x):
        return self.linear(self.embedding(x).mean(dim=-2))


def train_kaiming(
    filename: str = "reviews-train.txt",
    label: int = 0,
    n_vocab: int = 1024,
    embedding_dim: int = 64,
    lr: float = 0.001,
    batch_size: int = 16,
    n_epochs: int = 10,
):
    """Same as train() but uses ClassifierKaiming."""
    dataset = ReviewDataset(filename, label=label)
    processor = ReviewVectorizer(dataset, n_vocab)
    model = ClassifierKaiming(n_vocab, embedding_dim, len(processor.l2i))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    data_loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True, collate_fn=processor
    )
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0
        for bx, by in data_loader:
            optimizer.zero_grad()
            output = model(bx)
            loss = F.cross_entropy(output, by)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch}, loss: {running_loss / len(data_loader):.4f}")
    return processor, model


# Train with Kaiming init and compare against default initialisation
print("=== Kaiming Init: Category classifier ===")
torch.manual_seed(42)
vectorizer_cat_k, model_cat_k = train_kaiming(label=0)

print("\n=== Kaiming Init: Sentiment classifier ===")
torch.manual_seed(42)
vectorizer_sent_k, model_sent_k = train_kaiming(label=1)

In [ ]:
# Save Kaiming-initialised embeddings for Embedding Projector
save_embeddings(vectorizer_cat_k, model_cat_k, "vectors_cat_kaiming.tsv", "metadata_cat_kaiming.tsv")
save_embeddings(vectorizer_sent_k, model_sent_k, "vectors_sent_kaiming.tsv", "metadata_sent_kaiming.tsv")
print("Kaiming embedding files saved.")

**Analysis -- Kaiming vs. default (Normal) initialisation:**

The default `nn.Embedding` initialises weights from N(0,1). For a 64-dimensional embedding, this gives initial weight magnitudes of roughly 1.0, which can cause large activations early in training.

Kaiming uniform initialisation sets weights in the range `[-1/sqrt(fan_in), 1/sqrt(fan_in)]` (adjusted by slope `a`). For an embedding with `fan_in = num_embeddings = 1024`, this gives weights with a much smaller standard deviation, keeping the initial signal magnitudes in a regime where gradients flow more stably.

**Observed effects:**
- With Kaiming init, training typically converges slightly faster in the early epochs (lower loss at epochs 0-2) because the initial gradient signals are better scaled.
- Final loss after 10 epochs is often comparable between the two methods for this small dataset; the benefit of Kaiming init is more pronounced in deeper networks or with fewer training epochs.
- In the Embedding Projector, the Kaiming-initialised embedding space tends to show tighter initial clustering -- words start with smaller magnitudes and spread out more meaningfully as training progresses.

**🥳 Congratulations on finishing lab 1!**